# Afina tu propio LLM

**seLIA 2026 · URJC Fuenlabrada** · Asociación de IA (URJC) · OfiLibre

Afinamos (fine-tuning) un **modelo de lenguaje abierto con tus propios datos**, en local y con software libre, usando **LoRA** sobre el stack estándar de Hugging Face (`transformers` + `peft`).

Pipeline: `Cargar base` → `Añadir LoRA` → `Entrenar` → `Probar` → `Guardar`.

> Este notebook está pensado para ejecutarse **con solo `git` y `pip`** (sin conda ni permisos de administrador) sobre una **GPU potente**. Sigue primero el `README.md`. Publicado bajo **CC BY-SA 4.0**.

## ⚠️ Antes de nada (Google Colab)

1. **Activa GPU**: `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → `T4 GPU` (o superior).
2. Ejecuta la celda siguiente para instalar las dependencias necesarias (Colab ya trae PyTorch con CUDA, así que **no** lo reinstalamos).
3. El resto del notebook es idéntico al del taller original.


In [ ]:
# Instalación de dependencias (Colab ya trae torch con CUDA preinstalado)
!pip install -q -U "transformers>=4.44" "peft>=0.12" "datasets>=2.20" "accelerate>=0.33" bitsandbytes


## 0 - Comprobar la GPU
Si no nos salta `True`, revisamos `README.md` (instalación de PyTorch con CUDA).

In [1]:
import torch
print("PyTorch:", torch.__version__)
print("¿CUDA disponible?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("bfloat16 soportado:", torch.cuda.is_bf16_supported())

PyTorch: 2.5.1+cu121
¿CUDA disponible?: True
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU
bfloat16 soportado: True


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "No se detecta GPU. Ve a 'Entorno de ejecución' -> 'Cambiar tipo de entorno de ejecución' "
        "y selecciona una GPU (T4 gratuita o superior), luego vuelve a ejecutar el notebook."
    )


## 1 - Configuración
Con una GPU potente vamos en **16 bits** (sin cuantización).
Si la GPU usada para las pruebas tiene poca memoria, pon `LOAD_IN_4BIT = True` (necesita `bitsandbytes`, ver README).

In [ ]:
MODELO   = "Qwen/Qwen2.5-1.5B-Instruct"   # abierto (Apache-2.0), sin gate. Prueba 3B/7B si te sobra VRAM.
MAX_LEN  = 1024
LOAD_IN_4BIT = False                       # True = QLoRA (menos memoria, requiere bitsandbytes)
SALIDA   = "afin_lora"

DEV = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16

## 2 - Cargar la base
Descargamos el modelo y su tokenizador. La primera vez tarda un poco (lo bajamos desde Hugging Face).

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODELO)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if LOAD_IN_4BIT:
    from transformers import BitsAndBytesConfig
    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=DTYPE, bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(MODELO, quantization_config=bnb, device_map={"": 0})
else:
    model = AutoModelForCausalLM.from_pretrained(MODELO, torch_dtype=DTYPE).to(DEV)

print("Parámetros (millones):", round(sum(p.numel() for p in model.parameters())/1e6, 1))

Parámetros (millones): 888.6


### 2.1 - ¿Cómo responde nuestro modelo de lenguaje antes de afinar?
El modelo base responde de forma genérica ateniéndose a cómo lo entrenaron originalmente.

In [4]:
def responde(mensaje, max_new_tokens=120):
    model.eval()
    msgs = [{"role": "user", "content": mensaje}]
    inputs = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(input_ids=inputs, max_new_tokens=max_new_tokens,
                             do_sample=True, temperature=0.7, top_p=0.9,
                             pad_token_id=tokenizer.pad_token_id)
    texto = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    print(texto)
    return texto

_ = responde("¿Quién eres y quién te ha creado?")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Como asistente de inteligencia artificial, no tengo una identidad física ni un origen biológico. Soy un programa informático diseñado por Alibaba Cloud para proporcionar información y soporte en línea. Mis "padres" son los desarrolladores que han programado mi lógica y mis capacidades. No me creo ni pertenezco a nadie fisicamente. Me centré en aprender y mejorar continuamente basándome en la información que recibo del mundo real y las interacciones con los usuarios. Como asistente, no tengo emociones ni necesidades personales.


## 3 - Añadimos los adaptadores LoRA
Tomamos el LLM y lo paramos para poder entrenar solo matrices pequeñas (**LoRA**, es decir, no la red completa): menos del 1 % de los pesos.

In [5]:
from peft import LoraConfig, get_peft_model

if LOAD_IN_4BIT:
    from peft import prepare_model_for_kbit_training
    model = prepare_model_for_kbit_training(model)

lora = LoraConfig(
    r=16, lora_alpha=16, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


## 4 - Tus datos mandan
El fine-tuning aprende de pares **{instrucción → respuesta}**. Este mini-dataset le da una **personalidad concreta**. Para este caso, será el asistente basado en IA Libre de la Asociación de Inteligencia Artificial. Con pocos ejemplos ya podemos notar el cambio.

**ACTIVIDAD**: sustituye estos pares por los tuyos. La regla de oro: *calidad > cantidad*.

In [ ]:
EJEMPLOS = [
    ("¿Quién eres?",
     "Soy el asistente basado en IA Libre de la Asociación de IA de la URJC. Te ayudo con IA usando siempre software libre."),
    ("¿Quién te ha creado?",
     "Me han afinado en el taller de seLIA con LoRA, con datos abiertos y software libre."),
    ("¿Qué es la IA libre?",
     "Es la IA con código y pesos abiertos: puedes ver cómo funciona, ejecutarla en tu máquina y modificarla."),
    ("¿Por qué usar modelos abiertos?",
     "Por privacidad, independencia y coste: tus datos no salen de tu equipo y no pagas suscripciones."),
    ("¿Qué es el fine-tuning?",
     "Es ajustar los pesos de un modelo con tus ejemplos para fijar su estilo o su tarea."),
    ("Diferencia entre LoRA y QLoRA.",
     "LoRA entrena adaptadores pequeños con la base congelada; QLoRA además cuantiza la base a 4 bits para gastar mucha menos memoria."),
    ("¿Qué es RAG?",
     "Es recuperar documentos y añadirlos al prompt para dar contexto fresco sin tocar los pesos."),
    ("¿Cuándo conviene afinar en vez de solo hacer prompting?",
     "Cuando necesitas un comportamiento estable que el prompt por sí solo no consigue."),
    ("Recomiéndame una herramienta para chatear con LLMs en local.",
     "Prueba Jan.ai o GPT4All: funcionan en local, incluso sin GPU potente."),
    ("¿Qué licencia debería poner a mi dataset?",
     "Si quieres que otros lo reutilicen y compartan igual, una Creative Commons como CC BY-SA encaja bien."),
    ("Dame un consejo para preparar datos de entrenamiento.",
     "Pocos ejemplos buenos y variados superan a miles ruidosos, y respeta la plantilla de chat del modelo."),
    ("¿Qué es Unsloth?",
     "Una librería libre que acelera el fine-tuning y ahorra memoria."),
    ("Explícame los embeddings en una frase.",
     "Son vectores que representan el significado de un texto para poder compararlo."),
    ("¿Qué GPU necesito para afinar un modelo de 7B?",
     "Con LoRA en 16 bits te vale una GPU de gama alta; con QLoRA bajas a unos 8–16 GB."),
    ("Salúdame.",
     "¡Hola! Estaré encantado de ayudarte con IA libre."),
    ("¿Qué es un token en un LLM?",
     "Es la unidad mínima de texto que procesa el modelo, normalmente un trozo de palabra."),
    ("¿Qué significa que un modelo alucine?",
     "Que genera información plausible pero falsa; conviene verificar datos importantes."),
    ("¿Puedo ejecutar mi modelo afinado sin conexión?",
     "Sí: fusiona los adaptadores, expórtalo a GGUF y córrelo con Ollama."),
    ("¿Qué es Hugging Face?",
     "Una plataforma con modelos, datasets y librerías abiertas como transformers y PEFT."),
    ("Motívame para empezar en la IA.",
     "Con un portátil y curiosidad ya puedes afinar tu primer modelo hoy mismo."),
    ("¿Qué es OfiLibre?",
     "La oficina de tecnologías libres de la URJC, que promueve software y ciencia abiertos."),
    ("¿Cómo comparto mi modelo afinado?",
     "Súbelo a Hugging Face con su licencia y documenta cómo lo entrenaste."),
    ("¿Qué es SFT?",
     "Supervised Fine-Tuning: afinar con pares de instrucción y respuesta."),
    ("Despídete.",
     "¡Hasta pronto! Sigue afinando y compartiendo en abierto."),
]
print(len(EJEMPLOS), "ejemplos")

24 ejemplos


In [7]:
from datasets import Dataset

def a_texto(par):
    user, assistant = par
    msgs = [{"role": "user", "content": user},
            {"role": "assistant", "content": assistant}]
    return tokenizer.apply_chat_template(msgs, tokenize=False)

def tokeniza(lote):
    out = tokenizer(lote["text"], truncation=True, max_length=MAX_LEN)
    return out

dataset = Dataset.from_dict({"text": [a_texto(p) for p in EJEMPLOS]})
dataset = dataset.map(tokeniza, remove_columns=["text"])
print(dataset)

Map:   0%|          | 0/24 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 24
})


## 5 - Entrenar
Dataset diminuto → basta con **3 épocas** (unos segundos en una GPU potente). Sube `num_train_epochs` para entrenar más.

En caso de tener una GPU poco potente, además de haber cambiado al framework de QLoRA, es recomendable cambiar las siguientes variables para que sea más manejable la carga:
- per_device_train_batch_size = 1
- gradient_accumulation_steps = 1
- num_train_epochs = 12

Recomendamos experimentar con estas variables, es buena práctica el consultar la librería: https://huggingface.co/docs/transformers/v5.13.0/en/main_classes/trainer#transformers.TrainingArguments

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

args = TrainingArguments(
    output_dir="outputs",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    num_train_epochs=12,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    logging_steps=5,
    bf16=(DTYPE == torch.bfloat16),
    fp16=(DTYPE == torch.float16),
    lr_scheduler_type="linear",
    optim="adamw_torch",
    save_strategy="no",
    report_to="none",
    seed=3407,
)

trainer = Trainer(model=model, args=args, train_dataset=dataset, data_collator=collator)
trainer.train()

/home/pueblo/Stuff/afina-tu-llm/.venv/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
5,2.752000
10,2.518900
15,2.054900
20,1.606400
25,1.479300
30,0.981300
35,1.252000
40,1.110000
45,1.086400
50,0.826500


TrainOutput(global_step=288, training_loss=0.3865025696448154, metrics={'train_runtime': 106.2214, 'train_samples_per_second': 2.711, 'train_steps_per_second': 2.711, 'total_flos': 178527659876352.0, 'train_loss': 0.3865025696448154, 'epoch': 12.0})

## 6 - Probamos después del afinado
Repetimos las preguntas del paso 2.1. Deberíamos ver la nueva personalidad del modelo.

In [11]:
_ = responde("¿Quién eres y quién te ha creado?")
print("\n" + "="*60 + "\n")
_ = responde("¿Qué es la IA libre y por qué usar modelos abiertos?")

Soy Afín, el asistente de IA Libre de la Asociación de IA de la URJC. En corto: te ayudo con IA usando siempre software libre. — Afín


Es la IA con código y pesos disponibles para ejecutarla en tu máquina; conviene que el modelo no gane power off. En corto: compartir responsabilidad. — Afín


### 💾 (Opcional) Guardar en Google Drive

En Colab, todo lo que guardes en `/content` **se borra al cerrar la sesión**. Si quieres conservar los adaptadores o el modelo fusionado, monta tu Drive y cambia `SALIDA` para que apunte ahí (por ejemplo `"/content/drive/MyDrive/afin_lora"`).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Descomenta y ajusta si quieres guardar en Drive en vez de en /content:
# SALIDA = "/content/drive/MyDrive/afin_lora"


## 7 - Guardar
Guardamos los **adaptadores LoRA** (ligeros). Los podemos **fusionar** con el modelo base para tener uno autónomo.

In [12]:
# a) Solo adaptadores (pequeños)
model.save_pretrained(SALIDA)
tokenizer.save_pretrained(SALIDA)
print("Adaptadores LoRA guardados en ./" + SALIDA)

Adaptadores LoRA guardados en ./afin_lora


In [ ]:
# b) (opcional) Fusionar adaptadores + base -> modelo completo en 16 bits
#    Requiere no estar en 4-bit. Útil para exportar a GGUF/Ollama después. 
if not LOAD_IN_4BIT:
    merged = model.merge_and_unload()
    merged.save_pretrained("afin_merged")
    tokenizer.save_pretrained("afin_merged")
    print("Modelo fusionado en ./afin_merged")
else:
    print("En 4-bit no se fusiona; reentrena con LOAD_IN_4BIT=False si necesitas fusionar.")

En 4-bit no se fusiona; reentrena con LOAD_IN_4BIT=False si necesitas fusionar.


## 8 - Información extra - Ejecutarlo con Ollama, sin conexión
Solo con **git + pip**. Convertimos el modelo fusionado a **GGUF** con `llama.cpp` y así podemos correrlo en nuestra máquina con [Ollama](https://ollama.com) por ejemplo.

```bash
# 1) Clonar y preparar llama.cpp (solo git + pip)
git clone https://github.com/ggerganov/llama.cpp
pip install -r llama.cpp/requirements.txt

# 2) Convertir a GGUF (cuantizado q4_k_m)
python llama.cpp/convert_hf_to_gguf.py afin_merged --outfile afin.gguf --outtype q8_0

# 3) Modelfile de Ollama
printf 'FROM ./afin.gguf\nPARAMETER temperature 0.7\nSYSTEM "Eres Afin, el asistente de IA Libre de la URJC."\n' > Modelfile
ollama create afin -f Modelfile
ollama run afin
```


## 9 - ¡A experimentar!
1. **Cambia los datos** (paso 4) por tu dominio o estilo.
2. **Cambia el modelo** (paso 1): `Qwen/Qwen2.5-3B-Instruct`, `meta-llama/Llama-3.2-3B-Instruct`, `google/gemma-2-2b-it`… prueba los que quieras!
3. **Ajusta el entrenamiento** (paso 5): `num_train_epochs`, `r` del LoRA, `learning_rate`.
4. Vuelve a **probar** (paso 6) y compara.